In [1]:
import pandas as pd
import numpy as np

from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.PDB.PDBList import PDBList
from Bio.PDB.Polypeptide import *


In [2]:
def extract_sequences(data, chain_ids):
    """Return concatenated sequences for the specified chain IDs."""
    sequences = {}
    for chain_type, chain_id in chain_ids.items():
        if isinstance(chain_id, str):
            chain_id = [chain_id]

        entity_ids = []
        for i, strand_id_str in enumerate(data['_entity_poly.pdbx_strand_id']):
            strand_ids = [s.strip() for s in strand_id_str.split(',')]
            entity_id = data['_entity_poly.entity_id'][i]

            if any(strand_id in chain_id for strand_id in strand_ids):
                entity_ids.append(entity_id)

        full_sequence = ''
        for entity_id in entity_ids:
            try:
                index = data['_entity_poly.entity_id'].index(entity_id)
                sequence = data['_entity_poly.pdbx_seq_one_letter_code_can'][index]
                full_sequence += sequence.replace('\n', '')
            except ValueError:
                continue

        sequences[chain_type] = full_sequence

    return "|".join(sequences.values())


def extract_title(data):   
    """Return the PDB structure title.""" 
    return data['_struct.title']

In [ ]:
df = pd.read_table('../data/row/sabdab_summary_all.tsv')
df = df.query("antigen_type == 'peptide' or antigen_type == 'protein'")
df.reset_index(inplace=True)

In [ ]:
#Download structures

pdbl = PDBList()
pdb_id_list = df.loc[:, 'pdb'].str.strip().to_list()
directory = '../data/row/PDB_mmcif/'
for n in range(len(pdb_id_list)):
        try:
            pdb = pdb_id_list[n].upper()
            pdb_file = pdbl.download_pdb_files(pdb_id_list, pdir=directory)
        except Exception as e:
              continue

In [ ]:
for i in df.index: 
    name = df.loc[i, 'pdb']
    if type(df.loc[i, 'Hchain']) == str:
        Hchain = df.loc[i, 'Hchain']
    else:
        Hchain = 'None'
    
    if type(df.loc[i, 'Lchain']) == str:
        Lchain = df.loc[i, 'Lchain']
    else:
        Lchain = 'None'  
    antigen_chain = df.loc[i, 'antigen_chain']
    HL_chains_dict = {'Hchain': Hchain,
                      'Lchain': Lchain,
                      'antigen_chain': antigen_chain}
    
    # Load MMCIF file into dictionary
    pdb_dict = MMCIF2Dict(f"{directory}/{name}.cif")
    df.loc[i, 'HLA_chains_seq'] = extract_sequences(pdb_dict, HL_chains_dict)


In [ ]:
error_list = []

for i in df.index:
        name = df.loc[i, 'pdb']
        if type(df.loc[i, 'Hchain']) == str:
            Hchain = df.loc[i, 'Hchain']
        else:
            Hchain = 'None'
        
        if type(df.loc[i, 'Lchain']) == str:
            Lchain = df.loc[i, 'Lchain']
        else:
            Lchain = 'None'  
        antigen_chain = df.loc[i, 'antigen_chain']
        HL_chains_dict = {'Hchain': Hchain,
                        'Lchain': Lchain,
                        'antigen_chain': antigen_chain}
        
        try:
            # Load MMCIF file into dictionary
            pdb_dict = MMCIF2Dict(f"{directory}/{name}.cif")
            sequences = extract_sequences(pdb_dict, HL_chains_dict)
            df.loc[i, 'HLA_chains_seq'] = sequences
        except FileNotFoundError:
            print(f"Warning: File {name}.cif not found. Skipping entry.")
            error_list.append((i, name))
            df.loc[i, 'HLA_chains_seq'] = None
            continue
        except Exception as e:
            print(f"Error processing {name}: {str(e)}")
            error_list.append((i, name))
            df.loc[i, 'HLA_chains_seq'] = None
            continue


print("Error list:")
for error in error_list:
    print(f"Row {error[0]}, file {error[1]}")


In [8]:
#obsolete structures

updates = pd.Series({
    3194: '8RCQ',
    4394: np.nan,
    4724: '5m95',
    4767: '6FXN',
    4825: '5m95',
    4888: '5V2A',
    5026: '5m95',
    5391: '5WOB',
    5424: '5WOB',
    5510: '6IBL',
    5956: '3JBD',
    6257: '6FXN',
    6260: '6KS1',
    6282: '5WOB',
    6500: '5WOB',
    6619: '7P40',
    6688: '7P40',
    6696: '6B9J',
    6950: '6FXN',
    7002: np.nan,
    7051: '5VCN',
    7076: '5KQV',
    7093: '5WOB',
    7144: '6MXT',
    7229: '5VCN',
    7625: '6KS0',
    7643: '4ZXB',
    7660: '6FXN',
    7688: '5VPH',
    7770: '5VPG',
    8096: '4WEB',
    8278: '7R98',
    8283: '7ANQ',
    8300: '5VCO',
    8371: '4PS4',
    8390: np.nan,
    8534: '5UGY',
    8701: '5VCO',
    9052: '5WOB',
    9081: '3JBC',
    9220: '4ZXB',
    9260: '5WOB',
    9279: '5M94',
    9373: '6IBL',
    9576: '5UGY',
    9716: '4QEX',
    10029: '6NN3',
    10111: np.nan,
    10278: '5KQV',
    10582: '6FXN',
    10645: '4QEX',
    10672: '6FXN',
    10793: '4ZXB',
    10914: '6B9J',
    11219: '4ZXB',
    11338: '7P40',
    11357: '5WOB',
    11367: '5VPL',
    11459: '5UGY',
    11523: '7R98',
    11545: '7R98'
})

df.loc[updates.index, 'pdb'] = updates
error_indx = updates.keys()

In [ ]:
error_list_2 = []

for i in error_indx:
    if type(df.loc[i, 'pdb']) == str:
        name = df.loc[i, 'pdb'].lower()
        if type(df.loc[i, 'Hchain']) == str:
            Hchain = df.loc[i, 'Hchain']
        else:
            Hchain = 'None'
        
        if type(df.loc[i, 'Lchain']) == str:
            Lchain = df.loc[i, 'Lchain']
        else:
            Lchain = 'None'  
        antigen_chain = df.loc[i, 'antigen_chain']
        HL_chains_dict = {'Hchain': Hchain,
                        'Lchain': Lchain,
                        'antigen_chain': antigen_chain}
        
        try:
            # Load MMCIF file into dictionary
            pdb_dict = MMCIF2Dict(f"{directory}/{name}.cif")
            sequences = extract_sequences(pdb_dict, HL_chains_dict)
            df.loc[i, 'HLA_chains_seq'] = sequences
        except FileNotFoundError:
            print(f"Warning: File {name}.cif not found. Skipping entry.")
            error_list_2.append((i, name)) 
            df.loc[i, 'HLA_chains_seq'] = None
            continue
        except Exception as e:
            print(f"Error processing {name}: {str(e)}")
            error_list_2.append((i, name))
            df.loc[i, 'HLA_chains_seq'] = None
            continue


print("Error list:")
for error in error_list_2:
    print(f"Row {error[0]}, file {error[1]}")
    

In [10]:
split_sequences = df['HLA_chains_seq'].str.split('|', expand=True)

df['HL_chains_seq'] = split_sequences.iloc[:, 0].fillna('') + '|' + split_sequences.iloc[:, 1].fillna('')

df['antigen_chain_seq'] = split_sequences.iloc[:, 2]

In [ ]:
skipped_rows = []
for i in df.index:
    try:
        name = df.loc[i, 'pdb']
        pdb_dict = MMCIF2Dict(f"{directory}/{name}.cif")
        df.loc[i, 'annotation'] = extract_title(pdb_dict)
    except Exception as e:
        skipped_rows.append(i)

In [12]:
df['name'] = df[['pdb', 'Hchain', 'Lchain', 'antigen_chain']].apply(lambda x: '_'.join([str(i) for i in x if not pd.isnull(i)]), axis=1)

In [ ]:
df[['H_chain', 'L_chain']] = df['HL_chains_seq'].str.split('|', expand=True)

df['name_H'] = df['pdb'].astype(str) + '_' + df['Hchain'].astype(str) + '_H'
df['name_L'] = df['pdb'].astype(str) + '_' + df['Lchain'].astype(str) + '_L'

In [ ]:
df_h = df[['name_H', 'H_chain', 'antigen_name', 'antigen_chain_seq', 'annotation', 'antigen_type']]
df_h = df_h[df_h['H_chain'].notna() & (df_h['H_chain'] != '')]
df_h.drop_duplicates(subset=['name_H', 'H_chain', 'antigen_name', 'antigen_chain_seq'], inplace=True)
df_h.rename(columns={'name_H':'antibody_name', 'H_chain': 'antibody_sequence', 'antigen_name': 'target_name', 'antigen_chain_seq': 'target_seq'}, inplace=True)

df_l = df[['name_L', 'L_chain', 'antigen_name', 'antigen_chain_seq', 'annotation', 'antigen_type']]
df_l = df_l[df_l['L_chain'].notna() & (df_l['L_chain'] != '')]
df_l.drop_duplicates(subset=['name_L', 'L_chain', 'antigen_name', 'antigen_chain_seq'], inplace=True)
df_l.rename(columns={'name_L':'antibody_name', 'L_chain': 'antibody_sequence', 'antigen_name': 'target_name', 'antigen_chain_seq': 'target_seq'}, inplace=True)

In [ ]:
df_merge = pd.concat([df_h, df_l])
df_interactions = df_merge.drop(['annotation', 'antigen_type'], axis=1)
df_interactions.to_csv('../data/row/new_antibody_interactions.csv')

In [25]:
df_interactions['subtype'] = 'antibody'
df_interactions['representation_type'] = 'sequence'
df_interactions['type'] = 'protein'
df_interactions.rename(columns = {'antibody_name': 'name', 'antibody_sequence':'content'}, inplace = True)


In [ ]:
antibody_df = df_interactions[['name', 'type', 'subtype', 'representation_type', 'content', 'annotation']]
antibody_df.drop_duplicates(subset=['name', 'type', 'subtype', 'representation_type', 'content'], inplace=True)
antibody_df.to_csv('../data/row/antibodies.csv')

In [ ]:
molecules_df = df_merge[['target_name', 'target_seq', 'antigen_type']]

molecules_df['representation_type'] = molecules_df['antigen_type'].map({
    'small_molecule': 'smiles',
    'protein': 'sequence'
})

In [ ]:
molecules_df.rename(columns = {'target_name': 'name', 'target_seq':'content', 'antigen_type':'type'}, inplace = True)
molecules_df.drop_duplicates(inplace=True)
molecules_df.to_csv('../data/row/molecules_from_antibodies.csv')